## Evaluation plots

In [ ]:
from plotting_utils import get_last_x_metrics_files, plot_results

filepath_list = get_last_x_metrics_files(3)

# filepath_list = [
#     "runs/2026-03-05_22-28-05_v0.9.5-5-g631ed03_basic_same_route_straight100km_PPO/evaluation/metrics2026-03-06_12-18-04.json",
# ]

labels_to_include = None # Set to None to include all algorithms
# labels_to_include = ['GREEDY', 'PERFECT50', 'RANDOM']

metrics_to_plot = [
    "action_counts_absolute",
    "action_distribution",
    "termination_status",
    "reward",
    "env/ttt_per_ev_mean",
    "env/cwt_per_ev_mean",
    # "env/cumulated_waiting_time",
    "env/empty_vehicles_per_episode",
    "env/charging_stops_per_episode_mean",
    "episode_length",
    "env/final_simulation_time",
    "arrival_stats",
    "charging_start_stats",
    # "env/global_ttt",
    # "env/global_ttt_only_terminated",
    # "env/ttt_per_ev_mean_only_terminated",
    # "env/cumulated_waiting_time_only_terminated",
]

plot_results(filepath_list, metrics_to_plot, save_figure=True, algorithms_to_include=labels_to_include)

---

In [ ]:
from plotting_utils import plot_results_from_csv

plot_results_from_csv(
    "visuals/2026-05-21_22-16-44 allRandom BEST so far! (390k steps)/data_raw.csv",
    metrics_to_plot="all",
    algorithms_to_include=["BEST_GUESS", "GREEDY", "PPO_basic-simulation_time_pid1183697"],
    save_figure=True,
)

In [ ]:
# Get all unique algorithm names from a data_raw CSV file
import pandas as pd
print(pd.read_csv("visuals/2026-05-21_09-50-08 allRandom BEST so far! (180k steps)/data_raw.csv")["algorithm"].unique())

In [ ]:
from plotting_utils import generate_latex_tables

CSV_PATH = "visuals/2026-04-12_08-42-32 basicCongestion 20EV calibration (600k steps)/2026-04-27_12-39-31/data_raw.csv"
SAVE_PATH = "thesis/tables/experiment_1.1_results.tex"

generate_latex_tables(
    csv_path=CSV_PATH,
    metrics=[
        "action_counts_absolute",
        "action_distribution",
        "termination_status",
        "reward",
        "env/ttt_per_ev_mean",
        "env/cwt_per_ev_mean",
        "env/cumulated_waiting_time",
        "env/empty_vehicles_per_episode",
        "env/charging_stops_per_episode_mean",
        "episode_length",
        "env/final_simulation_time",
        "arrival_stats",
        "charging_start_stats",
        # "env/global_ttt",
        # "env/global_ttt_only_terminated",
        # "env/ttt_per_ev_mean_only_terminated",
        # "env/cumulated_waiting_time_only_terminated",
    ],
    algorithms_to_include=None,  # Set to None to include all algorithms
    save_path=SAVE_PATH,
)


---

## Test reward formulations

In [ ]:
# --- Visualize reward strategy on baseline algorithms ---

from functools import partial
from multiprocessing import Process

from training_utils import get_git_version, evaluate_model


BASE_EVAL = partial(evaluate_model,
        scenario="all_random",
        n_vehicles=20,
        reward_strategy="relativeDestinationCharging",
        # start_soc_bounds=(22_000, 22_000),
        version_tag=get_git_version(),
        street_network="straight_120km",
        n_noevs=0,
        n_episodes=10,
        # longest_route_duration=28_000,
        model_load_path=None,
        # render_mode="human",
        random_seed=54321,
)

EVALS = [
    partial(BASE_EVAL, algorithm="GREEDY"),
    partial(BASE_EVAL, algorithm="BEST_GUESS"),
    partial(BASE_EVAL, algorithm="RANDOM"),
]

def run_parallel(run_list):
    processes = [Process(target=run) for run in run_list]
    for p in processes:
        p.start()
    for p in processes:
        p.join()

run_parallel(EVALS)

from plotting_utils import get_last_x_metrics_files, plot_results

filepath_list = get_last_x_metrics_files(len(EVALS))

labels_to_include = None # Set to None to include all algorithms
# labels_to_include = ['GREEDY', 'PERFECT50', 'RANDOM']

metrics_to_plot = [
    "action_counts_absolute",
    # "action_distribution",
    "termination_status",
    "reward",
    "env/ttt_per_ev_mean",
    "env/cwt_per_ev_mean",
    # "env/cumulated_waiting_time",
    "env/empty_vehicles_per_episode",
    "env/charging_stops_per_episode_mean",
    "episode_length",
    "env/final_simulation_time",
    "arrival_stats",
    "charging_start_stats",
    # "env/global_ttt",
    # "env/global_ttt_only_terminated",
    # "env/ttt_per_ev_mean_only_terminated",
    # "env/cumulated_waiting_time_only_terminated",
]

plot_results(filepath_list, metrics_to_plot, save_figure=True, algorithms_to_include=labels_to_include)

---

# Old stuff left for reference:

In [ ]:
# Plot ratio of truncated episodes

import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# List of file paths
filepath_list = [
    "runs/2025-03-23_15-16-01_v0.7.5_shaping_circle_RANDOM/evaluation/metrics2025-03-23_15-16-00.json",
    "runs/2025-03-23_15-18-25_v0.7.5_shaping_circle_GREEDY/evaluation/metrics2025-03-23_15-18-24.json",
    "runs/2025-03-10_11-47-34_v0.7.5_shaping_circle_A2C/evaluation/metrics2025-03-23_15-16-43.json",
]

# Read all JSON files and combine the data into one list
data_list = []
for filepath in filepath_list:
    with open(filepath, 'r') as file:
        data_list.extend(json.load(file))

# Convert the combined data to a DataFrame
df = pd.DataFrame(data_list)

# Group by algorithm and calculate:
# - total episodes (by counting rows)
# - number of truncated episodes (summing the boolean column)
summary = df.groupby('algorithm').agg(
    total_episodes=('episode', 'count'),
    truncated_episodes=('was_truncated', 'sum')
).reset_index()

# Calculate the truncated ratio for each algorithm
summary['truncated_ratio'] = summary['truncated_episodes'] / summary['total_episodes']

print(summary)

# Plot the truncated ratio for each algorithm using seaborn
plt.figure(figsize=(8, 6))
sns.barplot(data=summary, x='algorithm', y='truncated_ratio')
plt.title("Fraction of Truncated Episodes per Algorithm")
plt.xlabel("Algorithm")
plt.ylabel("Fraction of Episodes Truncated")
plt.ylim(0, 1)  # Ratio between 0 and 1
plt.show()


In [ ]:
df[df["algorithm"] == "A2C"].head()

In [ ]:
# plot episode length

summary_ep_length = df.groupby('algorithm').agg(
    episode_length_mean=('episode_length', 'mean'),
    simulation_length_mean=('env/final_simulation_time', 'mean')
).reset_index()

summary_ep_length